# 13 — Automatic Polarimetric Mode Detection & Covariance Inventory

This is where the GCOV workflow hands off to polarimetric analysis. No new AOI gets created — it just reads back the product from Module 01, the frequency from Module 02, and the spatial_subset from Module 06.

Which branch activates depends on what's actually present in the GCOV datasets:
- HH, HV, VH, VV → LSAR full polarimetry
- RH, RV → SSAR compact/hybrid polarimetry

Only the modules matching the detected branch run; the rest print a skip message and move on.

**Masking:** every term Modules 14-19 load goes through the same GCOV `mask` layer Modules 07/08 use -- `MASK == 0` pixels are set to NaN in every channel, real and complex alike, before any reconstruction or decomposition runs. This matters specifically for compact-pol and full-pol acquisitions, which can have transmit-gap segments in the swath; those show up as MASK == 0 and are excluded automatically rather than contaminating the eigen/decomposition outputs with invalid data.

In [ ]:
from pathlib import Path
import h5py
import numpy as np
from nisar_utils.config import load_config
from nisar_utils.polarimetric_io import (detect_polarimetric_mode, save_detected_mode,
                                         get_authoritative_subset)

cfg=load_config(); path=Path(cfg['nisar_file'])
sp=get_authoritative_subset(cfg)
result=detect_polarimetric_mode(path,cfg.get('default_frequency'))
save_detected_mode(result)

def _read_scalar_metadata(f, path, default="not found"):
    try:
        obj=f[path]
        value=obj[()] if hasattr(obj, "shape") else obj
        if isinstance(value, np.ndarray) and value.size == 1:
            value=value.reshape(-1)[0]
        if isinstance(value, (bytes, np.bytes_)):
            value=value.decode("utf-8", errors="replace")
        if isinstance(value, np.generic):
            value=value.item()
        return value
    except (KeyError, OSError, TypeError, ValueError):
        return default

with h5py.File(path,'r') as f:
    meta_base=f"/science/{result['family']}/GCOV/metadata/processingInformation/parameters"
    is_full=_read_scalar_metadata(f, f"{meta_base}/isFullCovariance")
    sym=_read_scalar_metadata(f, f"{meta_base}/polarimetricSymmetrizationApplied")

print('Input:',path)
print('AUTHORITATIVE persisted subset:',sp)
print('Detected family:',result['family'])
print('Detected frequency:',result['frequency'])
print('Training branch:',result['mode'])
print('GCOV base:',result['base'])
print('Full covariance metadata:',is_full)
print('Polarimetric symmetrization metadata:',sym)
if result['mode']=='LSAR_FULL_POL':
    print('Analysis representation: C3 [HH, (HV+VH)/sqrt(2), VV] / standard Pauli T3')


In [ ]:
with h5py.File(path,'r') as f:
    names=sorted(f[result['base']].keys())
    info={k:(str(f[result['base']][k].dtype),tuple(f[result['base']][k].shape)) for k in names if k not in ('mask','numberOfLooks','rtcGammaToSigmaFactor')}
print('Available GCOV imagery layers:')
for k,v in info.items(): print(f'  {k:8s} {v}')
print('\nAutomatic routing:')
print('  -> Modules 14–17 :', 'ACTIVE' if result['mode']=='LSAR_FULL_POL' else 'SKIP')
print('  -> Module 18      :', 'ACTIVE' if result['mode']=='SSAR_COMPACT_POL' else 'SKIP')

### Training rule
Whatever subset just printed above is the one and only AOI for Modules 13 through 19. There's no mechanism to select a second one in this branch, and that's deliberate.